In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import sqlite3
from datetime import datetime

# Set database path in Google Drive
db_path = '/content/drive/MyDrive/expense_tracker_database.db'

def init_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Create category table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE
        )
    """)

    # Create expenses table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            amount REAL NOT NULL,
            category_id INTEGER NOT NULL,
            description TEXT,
            date TEXT NOT NULL,
            FOREIGN KEY (category_id) REFERENCES categories(id)
        )
    """)

    conn.commit()
    conn.close()

init_db()


In [11]:
def add_category(name):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO categories (name) VALUES (?)", (name,))
        conn.commit()
        print("Category added successfully.")
    except sqlite3.IntegrityError:
        print("Category already exists.")
    conn.close()


def list_categories():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT id, name FROM categories")
    categories = cursor.fetchall()
    if not categories:
        print("No categories found.")
    else:
        print("Categories:")
        for cat in categories:
            print(f"{cat[0]} - {cat[1]}")
    conn.close()


def add_expense(amount, category_id, description=""):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    date = datetime.now().strftime("%Y-%m-%d")

    # Check for duplicates (same amount, category, description, date)
    cursor.execute("""
        SELECT * FROM expenses
        WHERE amount = ? AND category_id = ? AND description = ? AND date = ?
    """, (amount, category_id, description, date))

    if cursor.fetchone():
        print("Duplicate expense already exists.")
    else:
        cursor.execute("""
            INSERT INTO expenses (amount, category_id, description, date)
            VALUES (?, ?, ?, ?)
        """, (amount, category_id, description, date))
        conn.commit()
        print("Expense recorded.")

    conn.close()


def show_expenses():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT e.id, e.amount, c.name, e.description, e.date
        FROM expenses e
        JOIN categories c ON e.category_id = c.id
        ORDER BY e.date DESC
    """)
    expenses = cursor.fetchall()
    if not expenses:
        print("No expenses found.")
    else:
        print("Expense History:")
        for exp in expenses:
            print(f"ID: {exp[0]} | Amount: {exp[1]} SAR | Category: {exp[2]} | Note: {exp[3]} | Date: {exp[4]}")
    conn.close()


def total_expenses():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(amount) FROM expenses")
    total = cursor.fetchone()[0]
    conn.close()
    print(f"Total Expenses: {total if total else 0:.2f} SAR")


In [12]:
# Add categories (only once)
add_category("Food")
add_category("Transport")
add_category("Coffee")
add_category("Shopping")

# List available categories
list_categories()

# Add new expenses (category ID must match list)
add_expense(35.0, 1, "Breakfast")
add_expense(12.5, 2, "Taxi fare")
add_expense(18.0, 3, "Coffee at Starbucks")

# Show all expenses
show_expenses()

# Show total
total_expenses()


Category added successfully.
Category added successfully.
Category added successfully.
Category added successfully.
Categories:
1 - Food
2 - Transport
3 - Coffee
4 - Shopping
Expense recorded.
Expense recorded.
Expense recorded.
Expense History:
ID: 1 | Amount: 35.0 SAR | Category: Food | Note: Breakfast | Date: 2025-09-27
ID: 2 | Amount: 12.5 SAR | Category: Transport | Note: Taxi fare | Date: 2025-09-27
ID: 3 | Amount: 18.0 SAR | Category: Coffee | Note: Coffee at Starbucks | Date: 2025-09-27
Total Expenses: 65.50 SAR


In [7]:
# Use it only if you want to start from scratch
def reset_database():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM expenses")
    cursor.execute("DELETE FROM categories")
    conn.commit()
    conn.close()
    print("Database has been cleared.")


In [13]:
# to Download the database file
from google.colab import files
files.download(db_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>